In [1]:
%reload_ext autoreload
%autoreload 2

In [18]:

import requests
import json

data_source = {
    "instruments": {
        "urls": ["https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/instruments/?format=json&page_num=1&page_size=2000",
                "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/instruments/?format=json&page_num=2&page_size=2000",]
    }
}

def get_json_data(url):
        try:
            # Fetch the data from the URL
            response = requests.get(url)
            response.raise_for_status()  # Raise an exception for bad status codes

            # Load the JSON data
            data = response.json()

            return data

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")


def get_pairs(urls):
    pairs = {} # key (short form): value [(long form)]
    for url in urls:
        data = get_json_data(url)
        if data:
            for item in data.get("concepts", []):
                sort_form = item.get("prefLabel")
                full_forms = [i.get("text") for i in item.get("definitions") if i.get("text")]
                if sort_form and full_forms:
                    pairs[sort_form] = full_forms

    return pairs

In [21]:
urls = data_source["instruments"]["urls"]
pairs = get_pairs(urls)

In [23]:
# checking if there are short form with more than one full form
_pairs = {k: v for k, v in pairs.items() if len(v) > 1} 
_pairs

{}

In [24]:
# hence changing the pairs structure from str: list(str)
pairs = {k: v[0] for k, v in _pairs.items() if len(v) > 0}